In [2]:
import sagemaker
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os
import boto3

sm_boto3 = boto3.client('sagemaker')
session = sagemaker.Session()
region = session.boto_region_name
bucket = 'sagemakerbucket-ml'
print("S3 bucket for saving data:" + bucket)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:8                                                                                    │
│                                                                                                  │
│    5 import os                                                                                   │
│    6 import boto3                                                                                │
│    7                                                                                             │
│ ❱  8 sm_boto3 = boto3.client('sagemaker')                                                        │
│    9 session = sagemaker.Session()                                                               │
│   10 region = session.boto_region_name                                                           │
│   11 bucket = 'sagemakerbucket-ml'                                                               │
│                                                                                                  │
│ c:\Users\user\Desktop\twitter-sentiment-sagemaker\myenv\Lib\site-packages\boto3\__init__.py:93   │
│ in client                                                                                        │
│                                                                                                  │
│    90 │                                                                                          │
│    91 │   See :py:meth:`boto3.session.Session.client`.                                           │
│    92 │   """                                                                                    │
│ ❱  93 │   return _get_default_session().client(*args, **kwargs)                                  │
│    94                                                                                            │
│    95                                                                                            │
│    96 def resource(*args, **kwargs):                                                             │
│                                                                                                  │
│ c:\Users\user\Desktop\twitter-sentiment-sagemaker\myenv\Lib\site-packages\boto3\session.py:337   │
│ in client                                                                                        │
│                                                                                                  │
│   334 │   │   │   # botocore version mismatches in AWS Lambda.                                   │
│   335 │   │   │   del create_client_kwargs['aws_account_id']                                     │
│   336 │   │                                                                                      │
│ ❱ 337 │   │   return self._session.create_client(                                                │
│   338 │   │   │   service_name, **create_client_kwargs                                           │
│   339 │   │   )                                                                                  │
│   340                                                                                            │
│                                                                                                  │
│ c:\Users\user\Desktop\twitter-sentiment-sagemaker\myenv\Lib\site-packages\botocore\context.py:12 │
│ 3 in wrapper                                                                                     │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)         

In [3]:
import sagemaker
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os
import boto3
from botocore.exceptions import NoRegionError

# ---- STEP 1: Set AWS region ----
region = 'us-east-1'  # N. Virginia
os.environ['AWS_DEFAULT_REGION'] = region

# ---- STEP 2: Create SageMaker + Boto3 sessions ----
try:
    boto_session = boto3.Session(region_name=region)
    sm_boto3 = boto_session.client('sagemaker')
    session = sagemaker.Session(boto_session=boto_session)
    print(f"✅ SageMaker session initialized in region: {region}")
except NoRegionError:
    print("❌ AWS region not set. Please configure region manually.")

# ---- STEP 3: Define S3 bucket ----
bucket = 'sagemakerbucket-ml'
print("✅ S3 bucket for saving data:", bucket)



✅ SageMaker session initialized in region: us-east-1
✅ S3 bucket for saving data: sagemakerbucket-ml


In [ ]:
# ---- STEP 4: Test access to the bucket ----
s3 = boto3.client('s3', region_name=region)
try:
    s3.head_bucket(Bucket=bucket)
    print(f"✅ Successfully connected to S3 bucket: {bucket}")
except Exception as e:
    print(f"⚠️ Could not access S3 bucket: {e}")


In [4]:
df = pd.read_csv(r"C:\Users\user\Desktop\twitter-sentiment-sagemaker\src\data\processed\reddit_features.csv")
print(df.head())

                                               title       flair  ...  wnat  work
0                          [d] self-promotion thread  Discussion  ...   0.0   0.0
1  [d] monthly who's hiring and who wants to be h...  Discussion  ...   0.0   0.0
2                        [d] on aaai 2026 discussion    Research  ...   0.0   0.0
3  push your creative models further. vast.ai han...         NaN  ...   0.0   0.0
4         [d] found error at published neurips paper    Research  ...   0.0   0.0

[5 rows x 117 columns]


In [5]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Columns: 117 entries, title to work
dtypes: float64(115), object(2)
memory usage: 20.2+ KB


In [6]:
df.columns

Index(['title', 'flair', '103m', '18pp', '2025', '2026', '70', 'aaai',
       'accuracy', 'actively',
       ...
       'tool', 'topics', 'tracing', 'trendy', 'variance', 'vast', 'vs',
       'wants', 'wnat', 'work'],
      dtype='object', length=117)

In [7]:
df.isnull().sum()

title    0
flair    2
103m     0
18pp     0
2025     0
        ..
vast     0
vs       0
wants    0
wnat     0
work     0
Length: 117, dtype: int64

In [9]:
# Fill missing flair values with "None"
df['flair'] = df['flair'].fillna("None")

# Verify
print("✅ Missing flair values handled:")
print(df['flair'].isnull().sum(), "missing flairs remaining")

✅ Missing flair values handled:
0 missing flairs remaining


In [10]:
df.to_csv("C:/Users/user/Desktop/twitter-sentiment-sagemaker/src/data/processed/reddit_features_clean.csv", index=False)
print("✅ Cleaned data saved successfully!")


✅ Cleaned data saved successfully!


In [11]:
# Separate features (X) and labels (y)
X = df.drop(columns=['flair', 'title'])
y = df['flair']

In [12]:
X.head()

,103m,18pp,2025,2026,70,aaai,accuracy,actively,agent,agentic,agents,ai,analysis,areas,beens,book,calling,commoditized,context,control,covariate,creative,credits,desktop,dgx,discussion,emerging,engineering,english,environment,error,evaluations,execution,experience,feedback,gcp,general,gpus,handles,hear,...,opik,optimization,outperforms,overlapping,paper,people,phd,plain,pro,promotion,published,purpose,pursued,push,quebec,reinforcement,research,review,right,rl,roles,schedule,scratch,self,sequences,shift,source,squeezed,students,thread,tool,topics,tracing,trendy,variance,vast,vs,wants,wnat,work
0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.57735,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.57735,0.0,0.0,0.0,0.0,0.0,0.57735,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.00000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.5,0.0,0.0
2,0.0,0.0,0.0,0.529278,0.0,0.599944,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.599944,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.00000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.286979,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.391076,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.391076,0.391076,0.0,...,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.00000,0.000000,0.0,0.0,0.391076,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.391076,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.541045,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.4321,0.0,0.0,0.0,0.0,0.00000,0.541045,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0


In [13]:
y.unique()

array(['Discussion', 'Research', 'None', 'Project'], dtype=object)

In [14]:
y.head()

0    Discussion
1    Discussion
2      Research
3          None
4      Research
Name: flair, dtype: object

In [15]:
# Split into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [16]:
trainX=pd.DataFrame(X_train)
trainX['flair']=y_train

testX=pd.DataFrame(X_test)
testX['flair']=y_test

In [17]:
trainX.to_csv("C:/Users/user/Desktop/twitter-sentiment-sagemaker/src/data/processed/train-V-1.csv", index=False)
testX.to_csv("C:/Users/user/Desktop/twitter-sentiment-sagemaker/src/data/processed/test-V-1.csv", index=False)

In [18]:
#send data to s3 bucket
sk_prefix="sagemaker/reddit-flair-classification/sklearncontainer"
trainpath = session.upload_data(
    path="C:/Users/user/Desktop/twitter-sentiment-sagemaker/src/data/processed/train-V-1.csv",
    bucket=bucket,
    key_prefix=sk_prefix
)

testpath = session.upload_data(
    path="C:/Users/user/Desktop/twitter-sentiment-sagemaker/src/data/processed/test-V-1.csv",
    bucket=bucket,
    key_prefix=sk_prefix
)

print("✅ Training data uploaded to S3:", trainpath)
print("✅ Testing data uploaded to S3:", testpath)

✅ Training data uploaded to S3: s3://sagemakerbucket-ml/sagemaker/reddit-flair-classification/sklearncontainer/train-V-1.csv
✅ Testing data uploaded to S3: s3://sagemakerbucket-ml/sagemaker/reddit-flair-classification/sklearncontainer/test-V-1.csv


In [ ]:
%%writefile script.py
import argparse
import os
import pandas as pd
import joblib
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder



if __name__ == "__main__":
    # ---------- 1. Parse SageMaker Arguments ----------
    parser = argparse.ArgumentParser()

    # Hyperparameters
    parser.add_argument("--n-estimators", type=int, default=100)
    parser.add_argument("--max-depth", type=int, default=None)
    parser.add_argument("--min-samples-split", type=int, default=2)
    parser.add_argument("--min-samples-leaf", type=int, default=1)

    # Directories and files
    parser.add_argument("--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR"))
    parser.add_argument("--train", type=str, default=os.environ.get("SM_CHANNEL_TRAIN", ""))
    parser.add_argument("--test", type=str, default=os.environ.get("SM_CHANNEL_TEST", ""))
    parser.add_argument("--train-file", type=str, default="train-V-1.csv")
    parser.add_argument("--test-file", type=str, default="test-V-1.csv")

    args, _ = parser.parse_known_args()

    # ---------- 2. Display environment info ----------
    print("📦 SKLearn Version:", sklearn.__version__)
    print("📦 Joblib Version:", joblib.__version__)
    print("📁 Training directory:", args.train)
    print("📁 Testing directory:", args.test)

    # ---------- 3. Read training and test data ----------
    print("[INFO] Reading data...\n")
    train_path = os.path.join(args.train, args.train_file)
    test_path = os.path.join(args.test, args.test_file)

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    print("✅ Train shape:", train_df.shape)
    print("✅ Test shape:", test_df.shape)

    # ---------- 4. Encode labels and prepare features ----------
    le = LabelEncoder()
    train_df["flair"] = le.fit_transform(train_df["flair"].astype(str))
    test_df["flair"] = le.transform(test_df["flair"].astype(str))

    X_train = train_df.drop(columns=["flair"], errors="ignore")
    y_train = train_df["flair"]
    X_test = test_df.drop(columns=["flair"], errors="ignore")
    y_test = test_df["flair"]

    print("Number of features in training:", X_train.shape[1])
    print("Number of features in testing:", X_test.shape[1])

    # ---------- 5. Train Random Forest ----------
    print("🚀 Training Random Forest Classifier...")
    model = RandomForestClassifier(
        n_estimators=args.n_estimators,
        max_depth=args.max_depth,
        min_samples_split=args.min_samples_split,
        min_samples_leaf=args.min_samples_leaf,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)

    # ---------- 6. Evaluate ----------
    print("📊 Generating classification report...\n")
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))

    # ---------- 7. Save trained model and encoder ----------
    os.makedirs(args.model_dir, exist_ok=True)
    model_path = os.path.join(args.model_dir, "model.joblib")
    encoder_path = os.path.join(args.model_dir, "label_encoder.joblib")
    joblib.dump(model, model_path)
    joblib.dump(le, encoder_path)
    print(f"✅ Model saved at: {model_path}")
    print(f"✅ LabelEncoder saved at: {encoder_path}")


# ---------- 8. SageMaker Inference Functions ----------
def model_fn(model_dir):
    """Load the model for inference"""
    model_path = os.path.join(model_dir, "model.joblib")
    encoder_path = os.path.join(model_dir, "label_encoder.joblib")
    print(f"[INFO] Loading model from {model_path}")
    model = joblib.load(model_path)
    le = joblib.load(encoder_path)
    return {"model": model, "encoder": le}

def input_fn(request_body, request_content_type):
    """Deserialize input data"""
    import io
    if request_content_type == "text/csv":
        df = pd.read_csv(io.StringIO(request_body), header=None)
        return df.fillna(0)
    else:
        raise ValueError(f"Unsupported content type: {request_content_type}")

def predict_fn(input_data, model_dict):
    """Make prediction"""
    model = model_dict["model"]
    preds = model.predict(input_data)
    return preds  # numeric predictions

def output_fn(prediction, response_content_type):
    """Serialize prediction output"""
    if response_content_type == "text/csv":
        return ",".join(map(str, prediction))
    else:
        raise ValueError(f"Unsupported content type: {response_content_type}")


Writing script.py


In [69]:
from sagemaker.sklearn.estimator import SKLearn

FRAMEWORK_VERSION = "0.23-1"

sklearn_estimator = SKLearn(
    entry_point="script.py",
    role="arn:aws:iam::375756730874:role/AmazonSageMaker-ExecutionRole",
    instance_count=1,
    instance_type="ml.m5.large",
    framework_version=FRAMEWORK_VERSION,
    base_job_name="RF-custom-sklearn",
    hyperparameters={
        "n_estimators": 100,
        "random_state": 0
    },
    use_spot_instances=True,
    max_wait=7200,     # Max time including spot delay
    max_run=3600       # Max time for the actual training job
)


In [70]:
sklearn_estimator.fit({"train": trainpath, "test": testpath}, wait=True)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: RF-custom-sklearn-2025-10-19-20-18-06-780


2025-10-19 20:18:47 Starting - Starting the training job..
2025-10-19 20:19:25 Downloading - Downloading input data..
2025-10-19 20:19:50 Downloading - Downloading the training image...
2025-10-19 20:20:46 Training - Training image download completed. Training in progress.
2025-10-19 20:20:46 Uploading - Uploading generated training model2025-10-19 20:20:40,369 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
2025-10-19 20:20:40,373 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2025-10-19 20:20:40,422 sagemaker_sklearn_container.training INFO     Invoking user training script.
2025-10-19 20:20:40,638 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2025-10-19 20:20:40,651 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2025-10-19 20:20:40,665 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2025-10-19 20:20:40,6

In [3]:
sklearn_estimator.latest_training_job.wait(logs="None")
artifact = sm_boto3.describe_training_job(
    TrainingJobName=sklearn_estimator.latest_training_job.name)["ModelArtifacts"]["S3ModelArtifacts"]
print("✅ Model artifacts located at:", artifact)

NameError: name 'sklearn_estimator' is not defined

In [72]:
artifact

's3://sagemaker-us-east-1-375756730874/RF-custom-sklearn-2025-10-19-20-18-06-780/output/model.tar.gz'

In [73]:
from sagemaker.sklearn.model import SKLearnModel
from time import gmtime, strftime

model_name = "Custom-sklearn-model-" + strftime("%Y-%m-%d-%H-%M-%S", gmtime())
model = SKLearnModel(
    name=model_name,
    model_data=artifact,
    role="arn:aws:iam::375756730874:role/AmazonSageMaker-ExecutionRole",
    entry_point="script.py",
    framework_version=FRAMEWORK_VERSION
)

In [74]:
model

In [75]:
model_name

'Custom-sklearn-model-2025-10-19-20-32-52'

In [2]:
endpoint_name = "Custom-sklearn-model-" + strftime("%Y-%m-%d-%H-%M-%S", gmtime())
print("EndpointName={}".format(endpoint_name))
predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.m4.xlarge",
    endpoint_name=endpoint_name
)

NameError: name 'strftime' is not defined

In [1]:
predictor

NameError: name 'predictor' is not defined

In [86]:
endpoint_name 

'Custom-sklearn-model-2025-10-19-20-33-33'

In [84]:
import pandas as pd

test_df = pd.read_csv(r"C:\Users\user\Desktop\twitter-sentiment-sagemaker\src\data\processed\train-V-1.csv")
test_df.head()

,103m,18pp,2025,2026,70,aaai,accuracy,actively,agent,agentic,agents,ai,analysis,areas,beens,book,calling,commoditized,context,control,covariate,creative,credits,desktop,dgx,discussion,emerging,engineering,english,environment,error,evaluations,execution,experience,feedback,gcp,general,gpus,handles,hear,...,optimization,outperforms,overlapping,paper,people,phd,plain,pro,promotion,published,purpose,pursued,push,quebec,reinforcement,research,review,right,rl,roles,schedule,scratch,self,sequences,shift,source,squeezed,students,thread,tool,topics,tracing,trendy,variance,vast,vs,wants,wnat,work,flair
0,0.0,0.306601,0.0,0.0,0.306601,0.0,0.306601,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.306601,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.306601,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.306601,0.0,0.0,0.0,0.0,0.306601,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.00000,0.306601,0.000000,0.0,0.000000,0.306601,0.0,0.0,0.0,0.000000,0.0,Research
1,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.00000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.5,0.000000,0.0,Discussion
2,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.281114,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.383084,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.305947,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.00000,0.000000,0.383084,0.0,0.383084,0.000000,0.0,0.0,0.0,0.000000,0.0,Discussion
3,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.57735,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.57735,0.0,0.0,0.0,0.0,0.0,0.57735,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,Discussion
4,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.464347,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.00000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.464347,0.0,Discussion


In [87]:
# Drop target columns and keep only numeric features
X_test = test_df.drop(columns=["flair",], errors="ignore")
X_test.head()

,103m,18pp,2025,2026,70,aaai,accuracy,actively,agent,agentic,agents,ai,analysis,areas,beens,book,calling,commoditized,context,control,covariate,creative,credits,desktop,dgx,discussion,emerging,engineering,english,environment,error,evaluations,execution,experience,feedback,gcp,general,gpus,handles,hear,...,opik,optimization,outperforms,overlapping,paper,people,phd,plain,pro,promotion,published,purpose,pursued,push,quebec,reinforcement,research,review,right,rl,roles,schedule,scratch,self,sequences,shift,source,squeezed,students,thread,tool,topics,tracing,trendy,variance,vast,vs,wants,wnat,work
0,0.0,0.306601,0.0,0.0,0.306601,0.0,0.306601,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.306601,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.306601,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.306601,0.0,0.0,0.0,0.0,0.306601,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.00000,0.306601,0.000000,0.0,0.000000,0.306601,0.0,0.0,0.0,0.000000,0.0
1,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.00000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.5,0.000000,0.0
2,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.281114,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.383084,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.305947,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.00000,0.000000,0.383084,0.0,0.383084,0.000000,0.0,0.0,0.0,0.000000,0.0
3,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.57735,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.57735,0.0,0.0,0.0,0.0,0.0,0.57735,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0
4,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.464347,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.00000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.464347,0.0


In [88]:


csv_input = X.head(5).to_csv(index=False, header=False)
response = predictor.predict(csv_input, initial_args={"ContentType": "text/csv"})
print(response)


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:2                                                                                    │
│                                                                                                  │
│   1 csv_input = X.head(5).to_csv(index=False, header=False)                                      │
│ ❱ 2 response = predictor.predict(csv_input, initial_args={"ContentType": "text/csv"})            │
│   3 print(response)                                                                              │
│   4                                                                                              │
│                                                                                                  │
│ c:\Users\user\Desktop\twitter-sentiment-sagemaker\myenv\Lib\site-packages\sagemaker\base_predict │
│ or.py:212 in predict                                                                             │
│                                                                                                  │
│   209 │   │   if inference_component_name:                                                       │
│   210 │   │   │   request_args["InferenceComponentName"] = inference_component_name              │
│   211 │   │                                                                                      │
│ ❱ 212 │   │   response = self.sagemaker_session.sagemaker_runtime_client.invoke_endpoint(**req   │
│   213 │   │   return self._handle_response(response)                                             │
│   214 │                                                                                          │
│   215 │   def _handle_response(self, response):                                                  │
│                                                                                                  │
│ c:\Users\user\Desktop\twitter-sentiment-sagemaker\myenv\Lib\site-packages\botocore\client.py:602 │
│ in _api_call                                                                                     │
│                                                                                                  │
│    599 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    600 │   │   │   │   )                                                                         │
│    601 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  602 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    603 │   │                                                                                     │
│    604 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    605                                                                                           │
│                                                                                                  │
│ c:\Users\user\Desktop\twitter-sentiment-sagemaker\myenv\Lib\site-packages\botocore\context.py:12 │
│ 3 in wrapper                                                                                     │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)                                               │
│   124 │   │                                                                                      │
│   125 │   │   return wrapper                                                                     │
│   126                                                      

In [89]:
import pandas as pd
from sagemaker.sklearn.model import SKLearnPredictor

# Load your CSV
X_test = pd.read_csv(r"C:\Users\user\Desktop\twitter-sentiment-sagemaker\src\data\processed\train-V-1.csv")
X_test = X_test.drop(columns=["flair"], errors="ignore")  # only features

# Make sure columns are sorted in the same order as training
X_test = X_test.reindex(sorted(X_test.columns), axis=1)

# Take a few rows
csv_input = X_test.head(5).to_csv(index=False, header=False)

# Use your deployed predictor
predictor = SKLearnPredictor(endpoint_name="Custom-sklearn-model-2025-10-19-20-33-33")
response = predictor.predict(csv_input, initial_args={"ContentType": "text/csv"})

# Decode
preds = response.decode("utf-8").strip().split(",")
print("Predicted labels (numeric):", preds)


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:16                                                                                   │
│                                                                                                  │
│   13                                                                                             │
│   14 # Use your deployed predictor                                                               │
│   15 predictor = SKLearnPredictor(endpoint_name="Custom-sklearn-model-2025-10-19-20-33-33")      │
│ ❱ 16 response = predictor.predict(csv_input, initial_args={"ContentType": "text/csv"})           │
│   17                                                                                             │
│   18 # Decode                                                                                    │
│   19 preds = response.decode("utf-8").strip().split(",")                                         │
│                                                                                                  │
│ c:\Users\user\Desktop\twitter-sentiment-sagemaker\myenv\Lib\site-packages\sagemaker\base_predict │
│ or.py:212 in predict                                                                             │
│                                                                                                  │
│   209 │   │   if inference_component_name:                                                       │
│   210 │   │   │   request_args["InferenceComponentName"] = inference_component_name              │
│   211 │   │                                                                                      │
│ ❱ 212 │   │   response = self.sagemaker_session.sagemaker_runtime_client.invoke_endpoint(**req   │
│   213 │   │   return self._handle_response(response)                                             │
│   214 │                                                                                          │
│   215 │   def _handle_response(self, response):                                                  │
│                                                                                                  │
│ c:\Users\user\Desktop\twitter-sentiment-sagemaker\myenv\Lib\site-packages\botocore\client.py:602 │
│ in _api_call                                                                                     │
│                                                                                                  │
│    599 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    600 │   │   │   │   )                                                                         │
│    601 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  602 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    603 │   │                                                                                     │
│    604 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    605                                                                                           │
│                                                                                                  │
│ c:\Users\user\Desktop\twitter-sentiment-sagemaker\myenv\Lib\site-packages\botocore\context.py:12 │
│ 3 in wrapper                                                                                     │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)         

In [ ]:
import boto3
import tarfile
import os
import joblib
import pandas as pd

# --- 1. Setup ---
s3_bucket = "sagemaker-us-east-1-375756730874"  # e.g., "sagemaker-your-bucket"
model_artifacts_key = "RF-custom-sklearn-2025-10-19-20-18-06-780/output/model.tar.gz"  # e.g., "sklearn/reddit-flair-classification/model.tar.gz"
local_dir = "model_local"

os.makedirs(local_dir, exist_ok=True)

# --- 2. Download model artifacts from S3 ---
s3 = boto3.client("s3")
#s3 = boto3.client("s3")
local_tar_path = os.path.join(local_dir, "model.tar.gz")
s3.download_file(s3_bucket, model_artifacts_key, local_tar_path)
print(f"✅ Downloaded model artifacts to {local_tar_path}")

# --- 3. Extract model and encoder ---
with tarfile.open(local_tar_path, "r:gz") as tar:
    tar.extractall(path=local_dir)
print(f"✅ Extracted artifacts to {local_dir}")

model_path = os.path.join(local_dir, "model.joblib")
encoder_path = os.path.join(local_dir, "label_encoder.joblib")

model = joblib.load(model_path)
le = joblib.load(encoder_path)
print("✅ Model and LabelEncoder loaded")

# --- 4. Load your CSV for prediction ---
data_path = "C:/Users/user/Desktop/twitter-sentiment-sagemaker/src/data/processed/train-V-1.csv"
df = pd.read_csv(data_path)

# Keep only numeric features, ensure same order as training
X = df.drop(columns=["flair"], errors="ignore")
X = X.reindex(sorted(X.columns), axis=1)  # important: keep column order consistent
X = X.fillna(0)  # handle missing values

# --- 5. Predict ---
pred_numeric = model.predict(X.head(5))
pred_labels = le.inverse_transform(pred_numeric)

print("Numeric predictions:", pred_numeric)
print("Decoded flair labels:", pred_labels)


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:18                                                                                   │
│                                                                                                  │
│   15 s3 = boto3.client("s3")                                                                     │
│   16 #s3 = boto3.client("s3")                                                                    │
│   17 local_tar_path = os.path.join(local_dir, "model.tar.gz")                                    │
│ ❱ 18 s3.download_file(s3_bucket, model_artifacts_key, local_tar_path)                            │
│   19 print(f"✅ Downloaded model artifacts to {local_tar_path}")                                 │
│   20                                                                                             │
│   21 # --- 3. Extract model and encoder ---                                                      │
│                                                                                                  │
│ c:\Users\user\Desktop\twitter-sentiment-sagemaker\myenv\Lib\site-packages\botocore\context.py:12 │
│ 3 in wrapper                                                                                     │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)                                               │
│   124 │   │                                                                                      │
│   125 │   │   return wrapper                                                                     │
│   126                                                                                            │
│                                                                                                  │
│ c:\Users\user\Desktop\twitter-sentiment-sagemaker\myenv\Lib\site-packages\boto3\s3\inject.py:223 │
│ in download_file                                                                                 │
│                                                                                                  │
│   220 │   │   transfer.                                                                          │
│   221 │   """                                                                                    │
│   222 │   with S3Transfer(self, Config) as transfer:                                             │
│ ❱ 223 │   │   return transfer.download_file(                                                     │
│   224 │   │   │   bucket=Bucket,                                                                 │
│   225 │   │   │   key=Key,                                                                       │
│   226 │   │   │   filename=Filename,                                                             │
│                                                                                                  │
│ c:\Users\user\Desktop\twitter-sentiment-sagemaker\myenv\Lib\site-packages\boto3\s3\transfer.py:4 │
│ 07 in download_file                                                                              │
│                                                                                                  │
│   404 │   │   │   bucket, key, filename, extra_args, subscribers                                 │
│   405 │   │   )                                                                                  │
│   406 │   │   try:                                                                               │
│ ❱ 407 │   │   │   future.result()                           

In [63]:
import pandas as pd

test_df = pd.read_csv(r"C:\Users\user\Desktop\twitter-sentiment-sagemaker\src\data\processed\train-V-1.csv")
test_df.head()

,103m,18pp,2025,2026,70,aaai,accuracy,actively,agent,agentic,agents,ai,analysis,areas,beens,book,calling,commoditized,context,control,covariate,creative,credits,desktop,dgx,discussion,emerging,engineering,english,environment,error,evaluations,execution,experience,feedback,gcp,general,gpus,handles,hear,...,optimization,outperforms,overlapping,paper,people,phd,plain,pro,promotion,published,purpose,pursued,push,quebec,reinforcement,research,review,right,rl,roles,schedule,scratch,self,sequences,shift,source,squeezed,students,thread,tool,topics,tracing,trendy,variance,vast,vs,wants,wnat,work,flair
0,0.0,0.306601,0.0,0.0,0.306601,0.0,0.306601,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.306601,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.306601,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.306601,0.0,0.0,0.0,0.0,0.306601,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.00000,0.306601,0.000000,0.0,0.000000,0.306601,0.0,0.0,0.0,0.000000,0.0,Research
1,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.00000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.5,0.000000,0.0,Discussion
2,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.281114,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.383084,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.305947,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.00000,0.000000,0.383084,0.0,0.383084,0.000000,0.0,0.0,0.0,0.000000,0.0,Discussion
3,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.57735,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.57735,0.0,0.0,0.0,0.0,0.0,0.57735,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,Discussion
4,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.464347,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.00000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.464347,0.0,Discussion


In [64]:
test_df.columns

Index(['103m', '18pp', '2025', '2026', '70', 'aaai', 'accuracy', 'actively',
       'agent', 'agentic',
       ...
       'topics', 'tracing', 'trendy', 'variance', 'vast', 'vs', 'wants',
       'wnat', 'work', 'flair'],
      dtype='object', length=116)

In [65]:
# Drop target columns and keep only numeric features
X_test = test_df.drop(columns=["flair",], errors="ignore")
X_test.head()

,103m,18pp,2025,2026,70,aaai,accuracy,actively,agent,agentic,agents,ai,analysis,areas,beens,book,calling,commoditized,context,control,covariate,creative,credits,desktop,dgx,discussion,emerging,engineering,english,environment,error,evaluations,execution,experience,feedback,gcp,general,gpus,handles,hear,...,opik,optimization,outperforms,overlapping,paper,people,phd,plain,pro,promotion,published,purpose,pursued,push,quebec,reinforcement,research,review,right,rl,roles,schedule,scratch,self,sequences,shift,source,squeezed,students,thread,tool,topics,tracing,trendy,variance,vast,vs,wants,wnat,work
0,0.0,0.306601,0.0,0.0,0.306601,0.0,0.306601,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.306601,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.306601,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.306601,0.0,0.0,0.0,0.0,0.306601,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.00000,0.306601,0.000000,0.0,0.000000,0.306601,0.0,0.0,0.0,0.000000,0.0
1,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.00000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.5,0.000000,0.0
2,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.281114,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.383084,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.305947,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.00000,0.000000,0.383084,0.0,0.383084,0.000000,0.0,0.0,0.0,0.000000,0.0
3,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.57735,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.57735,0.0,0.0,0.0,0.0,0.0,0.57735,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0
4,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.464347,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.00000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.464347,0.0


In [66]:
#import pandas as pd

#test_df = pd.read_csv(r"C:\Users\user\Desktop\twitter-sentiment-sagemaker\src\data\processed\train-V-1.csv")

# Drop target columns and keep only numeric features
#X_test = test_df.drop(columns=["flair",], errors="ignore")

# Ensure same feature order as during training
#X_test = X_test.reindex(sorted(X_test.columns), axis=1)

# Prepare small sample (no header, no index)
csv_input = X_test.head(5).to_csv(index=False, header=False)

# Make prediction
response = predictor.predict(csv_input, initial_args={"ContentType": "text/csv"})
print("Raw response:", response)

# Decode
try:
    preds = response.decode("utf-8").strip().split(",")
    print("Predicted flair labels:", preds)
except Exception as e:
    print("⚠️ Decode error:", e)


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:15                                                                                   │
│                                                                                                  │
│   12 csv_input = X_test.head(5).to_csv(index=False, header=False)                                │
│   13                                                                                             │
│   14 # Make prediction                                                                           │
│ ❱ 15 response = predictor.predict(csv_input, initial_args={"ContentType": "text/csv"})           │
│   16 print("Raw response:", response)                                                            │
│   17                                                                                             │
│   18 # Decode                                                                                    │
│                                                                                                  │
│ c:\Users\user\Desktop\twitter-sentiment-sagemaker\myenv\Lib\site-packages\sagemaker\base_predict │
│ or.py:212 in predict                                                                             │
│                                                                                                  │
│   209 │   │   if inference_component_name:                                                       │
│   210 │   │   │   request_args["InferenceComponentName"] = inference_component_name              │
│   211 │   │                                                                                      │
│ ❱ 212 │   │   response = self.sagemaker_session.sagemaker_runtime_client.invoke_endpoint(**req   │
│   213 │   │   return self._handle_response(response)                                             │
│   214 │                                                                                          │
│   215 │   def _handle_response(self, response):                                                  │
│                                                                                                  │
│ c:\Users\user\Desktop\twitter-sentiment-sagemaker\myenv\Lib\site-packages\botocore\client.py:602 │
│ in _api_call                                                                                     │
│                                                                                                  │
│    599 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    600 │   │   │   │   )                                                                         │
│    601 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  602 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    603 │   │                                                                                     │
│    604 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    605                                                                                           │
│                                                                                                  │
│ c:\Users\user\Desktop\twitter-sentiment-sagemaker\myenv\Lib\site-packages\botocore\context.py:12 │
│ 3 in wrapper                                                                                     │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)         

In [50]:
import pandas as pd

df = pd.read_csv(r"C:\Users\user\Desktop\twitter-sentiment-sagemaker\src\data\processed\reddit_features_clean.csv")

# Drop non-feature columns
X = df.drop(columns=["flair", "title"], errors='ignore')

# Verify all are numeric
print("Columns:", X.columns.tolist())
print("Any non-numeric?", not X.applymap(lambda x: isinstance(x, (int, float))).all().all())


Columns: ['103m', '18pp', '2025', '2026', '70', 'aaai', 'accuracy', 'actively', 'agent', 'agentic', 'agents', 'ai', 'analysis', 'areas', 'beens', 'book', 'calling', 'commoditized', 'context', 'control', 'covariate', 'creative', 'credits', 'desktop', 'dgx', 'discussion', 'emerging', 'engineering', 'english', 'environment', 'error', 'evaluations', 'execution', 'experience', 'feedback', 'gcp', 'general', 'gpus', 'handles', 'hear', 'heating', 'hired', 'hiring', 'house', 'iclr', 'implementation', 'improve', 'industry', 'internal', 'interview', 'interviewers', 'jobs', 'json', 'learning', 'level', 'like', 'llm', 'llms', 'looking', 'mac', 'mila', 'minimax', 'ml', 'mle', 'modelling', 'models', 'moe', 'monthly', 'moving', 'multi', 'neurips', 'nlp', 'numerical', 'nvidia', 'open', 'opik', 'optimization', 'outperforms', 'overlapping', 'paper', 'people', 'phd', 'plain', 'pro', 'promotion', 'published', 'purpose', 'pursued', 'push', 'quebec', 'reinforcement', 'research', 'review', 'right', 'rl', 'rol

C:\Users\user\AppData\Local\Temp\ipykernel_17844\3590976387.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  print("Any non-numeric?", not X.applymap(lambda x: isinstance(x, (int, float))).all().all())


Any non-numeric? False


In [51]:
csv_input = X.head(5).to_csv(index=False, header=False)


In [52]:
print("CSV Input:\n", csv_input)

CSV Input:
 0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5773502691896258,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5773502691896258,0.0,0.0,0.0,0.0,0.0,0.5773502691896258,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.0
0.0,0.0,0.0,0.529277898

In [53]:
print("Number of features in CSV:", X.shape[1])


Number of features in CSV: 115


In [54]:
import boto3
import joblib
import os

s3 = boto3.client("s3")

# Path from when you deployed your model
# Example: "s3://your-bucket-name/sagemaker/sklearn-model/model.tar.gz"
model_s3_path = "s3://sagemaker-us-east-1-375756730874/RF-custom-sklearn-2025-10-19-17-53-58-999/output/model.tar.gz"

# Split bucket and key
path_parts = model_s3_path.replace("s3://", "").split("/", 1)
bucket_name = path_parts[0]
key = path_parts[1]

# Download locally
os.makedirs("model_dir", exist_ok=True)
local_path = "model_dir/model.tar.gz"
s3.download_file(bucket_name, key, local_path)

print("✅ Model downloaded to:", local_path)


✅ Model downloaded to: model_dir/model.tar.gz


In [55]:
import tarfile

# Extract model archive
with tarfile.open(local_path, "r:gz") as tar:
    tar.extractall(path="model_dir")

# Find the joblib file
import glob
joblib_path = glob.glob("model_dir/**/*.joblib", recursive=True)[0]
print("Found model file:", joblib_path)

# Load it
model = joblib.load(joblib_path)
print("✅ Model loaded successfully.")


C:\Users\user\AppData\Local\Temp\ipykernel_17844\1501646398.py:5: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path="model_dir")


Found model file: model_dir\model.joblib


c:\Users\user\Desktop\twitter-sentiment-sagemaker\myenv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 0.23.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:13                                                                                   │
│                                                                                                  │
│   10 print("Found model file:", joblib_path)                                                     │
│   11                                                                                             │
│   12 # Load it                                                                                   │
│ ❱ 13 model = joblib.load(joblib_path)                                                            │
│   14 print("✅ Model loaded successfully.")                                                      │
│   15                                                                                             │
│                                                                                                  │
│ c:\Users\user\Desktop\twitter-sentiment-sagemaker\myenv\Lib\site-packages\joblib\numpy_pickle.py │
│ :749 in load                                                                                     │
│                                                                                                  │
│   746 │   │   │   │   # A memory-mapped array has to be mapped with the endianness               │
│   747 │   │   │   │   # it has been written with. Other arrays are coerced to the                │
│   748 │   │   │   │   # native endianness of the host system.                                    │
│ ❱ 749 │   │   │   │   obj = _unpickle(                                                           │
│   750 │   │   │   │   │   fobj,                                                                  │
│   751 │   │   │   │   │   ensure_native_byte_order=ensure_native_byte_order,                     │
│   752 │   │   │   │   │   filename=filename,                                                     │
│                                                                                                  │
│ c:\Users\user\Desktop\twitter-sentiment-sagemaker\myenv\Lib\site-packages\joblib\numpy_pickle.py │
│ :626 in _unpickle                                                                                │
│                                                                                                  │
│   623 │   )                                                                                      │
│   624 │   obj = None                                                                             │
│   625 │   try:                                                                                   │
│ ❱ 626 │   │   obj = unpickler.load()                                                             │
│   627 │   │   if unpickler.compat_mode:                                                          │
│   628 │   │   │   warnings.warn(                                                                 │
│   629 │   │   │   │   "The file '%s' has been generated with a "                                 │
│                                                                                                  │
│ C:\Program Files\Python313\Lib\pickle.py:1256 in load                                            │
│                                                                                                  │
│   1253 │   │   │   │   if not key:                                                               │
│   1254 │   │   │   │   │   raise EOFError                                                        │
│   1255 │   │   │   │   assert isinstance(key, bytes_types)                                       │
│ ❱ 1256 │   │   │   │   dispatch[key[0]](self)                                                    │
│   1257 │   │   except _Stop as stopinst:                                                         │
│   1258 │   │   │   return stopinst.value                    

In [5]:
import tarfile
import os

tar_path = r"C:\Users\user\Desktop\twitter-sentiment-sagemaker\model.tar.gz"
extract_dir = r"C:\Users\user\Desktop\twitter-sentiment-sagemaker\artifacts"

os.makedirs(extract_dir, exist_ok=True)

with tarfile.open(tar_path, "r:gz") as tar:
    tar.extractall(path=extract_dir)

print("Extracted files:", os.listdir(extract_dir))


Extracted files: ['label_encoder.joblib', 'model.joblib']


C:\Users\user\AppData\Local\Temp\ipykernel_10136\1580076009.py:10: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_dir)


In [6]:
import joblib
import os

model_path = os.path.join(extract_dir, "model.joblib")
encoder_path = os.path.join(extract_dir, "label_encoder.joblib")

model = joblib.load(model_path)
label_encoder = joblib.load(encoder_path)

print("Model and encoder loaded successfully!")


c:\Users\user\Desktop\twitter-sentiment-sagemaker\myenv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 0.23.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


ValueError: node array from the pickle has an incompatible dtype:
- expected: {'names': ['left_child', 'right_child', 'feature', 'threshold', 'impurity', 'n_node_samples', 'weighted_n_node_samples', 'missing_go_to_left'], 'formats': ['<i8', '<i8', '<i8', '<f8', '<f8', '<i8', '<f8', 'u1'], 'offsets': [0, 8, 16, 24, 32, 40, 48, 56], 'itemsize': 64}
- got     : [('left_child', '<i8'), ('right_child', '<i8'), ('feature', '<i8'), ('threshold', '<f8'), ('impurity', '<f8'), ('n_node_samples', '<i8'), ('weighted_n_node_samples', '<f8')]